# PHẦN 2: THUẬT TOÁN FLAJOLET-MARTIN (1 HASH FUNCTION)
## Chuyên đề Xử Lý Dữ Liệu Lớn - Giảng viên: Trần Thị Nhi

### 1. Cơ sở lý thuyết
- **Mục tiêu:** Ước lượng số lượng phần tử duy nhất ($F_0$) trong luồng dữ liệu mà không cần lưu trữ toàn bộ các phần tử vào bộ nhớ RAM.
- **Nguyên lý cốt lõi:**
  - Sử dụng một hàm băm đồng nhất $h(x)$ ánh xạ mỗi phần tử sang một chuỗi nhị phân ngẫu nhiên (32-bit hoặc 64-bit).
  - Xác suất để một giá trị băm có $r$ bit 0 liên tiếp ở tận cùng (trailing zeros) là $2^{-(r+1)}$.
  - Theo dõi giá trị $R = \max(\text{trailing\_zeros})$ lớn nhất từng quan sát được qua luồng.
  - Công thức ước lượng Flajolet-Martin:
    $$E = \frac{2^R}{\phi}$$
    *(trong đó $\phi \approx 0.77351$ là hằng số hiệu chỉnh bias của Flajolet-Martin)*.
- **Ưu điểm:** Bộ nhớ siêu nhỏ $O(1)$ (chỉ cần 1 biến lưu số nguyên $R$), tính toán cực nhanh $O(1)$ cho mỗi phần tử.
- **Nhược điểm:** Sai số phương sai (variance) cao vì $R$ chỉ là số nguyên, khiến giá trị ước lượng nhảy vọt theo lũy thừa của 2 (đồ thị bậc thang).

In [ ]:
# 0. Kết nối Google Drive trên Google Colab
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    print("Đang kết nối tới Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive đã được kết nối sẵn!")

In [ ]:
# 1. Khởi tạo và nạp các thư viện cần thiết
import os
import sys
import time
import json
import hashlib
import tracemalloc
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
print("Đã nạp xong thư viện thành công!")

In [ ]:
# 2. Cấu hình đường dẫn dữ liệu và thư mục lưu trữ kết quả trên Google Drive
LOG_FILE = "/content/drive/MyDrive/accessLog/access.log"
RESULTS_DIR = "/content/drive/MyDrive/accessLog/results"

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Đường dẫn file log: {LOG_FILE}")
print(f"Thư mục lưu kết quả: {RESULTS_DIR}")

In [ ]:
# 3. Các hàm băm và đếm số bit 0 tận cùng
def count_trailing_zeros(n):
    """
    Đếm số lượng bit 0 liên tiếp ở tận cùng chuỗi nhị phân của số nguyên n.
    Sử dụng toán tử bit-shift để đạt tốc độ tối đa O(1).
    """
    if n == 0:
        return 32  # Giới hạn 32-bit
    zeros = 0
    while (n & 1) == 0:
        zeros += 1
        n >>= 1
    return zeros

def hash_function(data_str, seed=42):
    """
    Tạo ra hàm băm đồng nhất bằng cách băm chuỗi 'seed_data' qua MD5.
    Trả về số nguyên 32-bit không âm.
    """
    salted_input = f"{seed}_{data_str}".encode('utf-8')
    hash_digest = hashlib.md5(salted_input).hexdigest()
    return int(hash_digest[:8], 16)

print("Đã định nghĩa xong hàm băm và đếm bit 0!")

In [ ]:
# 4. Cài đặt lớp Thuật toán Flajolet-Martin (1 Hash)
class FlajoletMartin:
    """
    Thuật toán Flajolet-Martin (Sử dụng 1 hàm băm)
    """
    def __init__(self, seed=42):
        self.seed = seed
        self.max_zeros = 0
        self.phi = 0.77351  # Hằng số hiệu chỉnh Flajolet-Martin

    def update(self, item):
        """Cập nhật khi có phần tử mới đi qua luồng"""
        hash_val = hash_function(item, self.seed)
        r = count_trailing_zeros(hash_val)
        if r > self.max_zeros:
            self.max_zeros = r

    def estimate(self):
        """Ước lượng số lượng phần tử duy nhất F_0 = 2^R / phi"""
        return (2 ** self.max_zeros) / self.phi

# Tạo alias để tương thích ngược
FlajoletMartinBasic = FlajoletMartin
print("Đã cài đặt xong lớp FlajoletMartin (1 Hash)!")

In [ ]:
# 5. Hàm đọc luồng dữ liệu (Streaming Generator)
def stream_log_file(file_path):
    if not os.path.exists(file_path):
        print(f"[CẢNH BÁO] Không tìm thấy file: {file_path}")
        return
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line:
                yield line.split(' ', 1)[0]

In [ ]:
# 6. Hàm thực nghiệm Flajolet-Martin (1 Hash)
def run_fm_experiment(log_file_path, sample_step=50000, max_lines=None):
    fm = FlajoletMartin(seed=42)
    stream_counts = []
    estimates_history = []
    max_zeros_history = []

    # Nạp ground truth từ set_metrics.json trên Google Drive nếu có để đối chiếu
    exact_lookup = {}
    set_file = os.path.join(RESULTS_DIR, 'set_metrics.json')

    if os.path.exists(set_file):
        with open(set_file, 'r', encoding='utf-8') as f:
            set_data = json.load(f)
            for s, u in zip(set_data.get('history_steps', []), set_data.get('history_unique', [])):
                exact_lookup[s] = u

    print("="*70)
    print("BẮT ĐẦU THỰC NGHIỆM FLAJOLET-MARTIN (1 HASH)")
    print(f"File: {log_file_path} | Sample step: {sample_step:,} dòng")
    print("="*70)

    tracemalloc.start()
    start_time = time.time()
    total_processed = 0

    for ip in stream_log_file(log_file_path):
        total_processed += 1
        fm.update(ip)

        if total_processed % sample_step == 0:
            est = fm.estimate()
            stream_counts.append(total_processed)
            estimates_history.append(est)
            max_zeros_history.append(fm.max_zeros)

            exact_str = f"{exact_lookup[total_processed]:,}" if total_processed in exact_lookup else "Chưa có"
            print(f"Dòng: {total_processed:>10,} | Bit 0 max (R): {fm.max_zeros:>2} | Ước lượng FM: {int(est):>10,} | Thực tế: {exact_str}")

        if max_lines and total_processed >= max_lines:
            break

    elapsed_time = time.time() - start_time
    _, peak_ram = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    final_estimate = fm.estimate()
    fm_bytes = sys.getsizeof(fm) + sys.getsizeof(fm.max_zeros)

    print("\n" + "="*70)
    print("KẾT QUẢ TỔNG KẾT (FM 1 HASH):")
    print(f"- Tổng số dòng log đã quét:            {total_processed:,}")
    print(f"- Số bit 0 tận cùng lớn nhất (R):      {fm.max_zeros}")
    print(f"- Số IP ước lượng (2^R / phi):         {int(final_estimate):,}")
    print(f"- Thời gian thực thi:                  {elapsed_time:.2f} giây")
    print(f"- Dung lượng cấu trúc FM chiếm dụng:   {fm_bytes} Bytes ({fm_bytes/1024:.2f} KB)")
    print(f"- Peak RAM hệ thống:                   {peak_ram / (1024*1024):.4f} MB")
    print("="*70)

    # Lưu kết quả trực tiếp vào Google Drive (đồng thời ghi cả 2 tên tệp để đồng bộ)
    out_path = os.path.join(RESULTS_DIR, 'fm_metrics.json')
    out_path_compat = os.path.join(RESULTS_DIR, 'fm_basic_metrics.json')
    results_data = {
        "method": "Flajolet-Martin (1 Hash)",
        "total_processed": total_processed,
        "max_zeros": fm.max_zeros,
        "estimate_final": round(final_estimate, 2),
        "elapsed_time": round(elapsed_time, 2),
        "fm_bytes": fm_bytes,
        "peak_ram_bytes": peak_ram,
        "history_steps": stream_counts,
        "history_estimates": [round(e, 2) for e in estimates_history],
        "history_max_zeros": max_zeros_history
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, indent=2, ensure_ascii=False)
    with open(out_path_compat, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, indent=2, ensure_ascii=False)
    print(f"-> Đã lưu kết quả thành công tại Google Drive: '{out_path}'!")

    return results_data

In [ ]:
# 7. Chạy thực nghiệm
fm_results = run_fm_experiment(LOG_FILE, sample_step=50000)

In [ ]:
# 8. Trực quan hóa kết quả đọc trực tiếp từ tệp fm_metrics.json trên Google Drive
res_file = os.path.join(RESULTS_DIR, 'fm_metrics.json')
if not os.path.exists(res_file):
    res_file = os.path.join(RESULTS_DIR, 'fm_basic_metrics.json')

if os.path.exists(res_file):
    with open(res_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    steps = data['history_steps']
    estimates = data['history_estimates']
    r_values = data['history_max_zeros']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Đồ thị 1: Đường ước lượng FM dạng bậc thang
    ax1.step(steps, estimates, 'r-', where='post', linewidth=2, label='Ước lượng FM (1 Hash)')
    ax1.set_title(f'Ước lượng FM 1 Hash (Bước nhảy lũy thừa 2, Ước lượng: {int(data["estimate_final"]):,})', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Số dòng log đã xử lý')
    ax1.set_ylabel('Số lượng IP ước lượng')
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.legend()

    # Đồ thị 2: Diễn biến của chỉ số R (max bit 0)
    ax2.plot(steps, r_values, 'g-s', markersize=4, label='R (Max Trailing Zeros)')
    ax2.set_title(f'Tăng trưởng của Giá trị Bit 0 Lớn nhất R (Hiện tại R = {data["max_zeros"]})', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Số dòng log đã xử lý')
    ax2.set_ylabel('Giá trị R')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print(f"Chưa tìm thấy tệp '{res_file}'. Vui lòng chạy Cell 7 trước!")

### 9. Nhận xét và Phân tích Thuật toán Flajolet-Martin (1 Hash)
- **Hiệu quả bộ nhớ:** Cực kỳ vượt trội! Thuật toán chỉ tốn chưa tới $100$ Bytes bộ nhớ để theo dõi luồng hàng triệu IP, tiết kiệm gấp hàng chục ngàn lần so với cấu trúc `Set`.
- **Hạn chế lớn về độ mượt mà:** Do chỉ có một hàm băm, ước lượng $\hat{n} = 2^R / \phi$ chỉ có thể nhận các giá trị rời rạc (nhảy vọt gấp đôi mỗi khi tìm thấy một giá trị hash có thêm 1 bit 0). Điều này tạo ra đồ thị dạng bậc thang với độ lệch chuẩn rất cao (sai số lý thuyết $\approx 78\%$).
- **Hướng giải quyết:** Cần áp dụng kỹ thuật **PCSA 1-Hash Chia Bit kết hợp Median of Means** (được cài đặt ở Notebook `3_FM_PCSA.ipynb`) để làm phẳng đường ước lượng và triệt tiêu sai số.